In [8]:
import pandas as pd

movies = pd.read_csv("../server/data/ml-latest-small/movies.csv")
ratings = pd.read_csv("../server/data/ml-latest-small/ratings.csv")
tags = pd.read_csv("../server/data/ml-latest-small/tags.csv")
print(movies.head())
print(ratings.head())
print(tags.head())

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931
   userId  movieId              tag   timestamp
0       2    60756            funny  1445714994
1       2    60756  Highly quotable  1445714996
2       2    60756     will ferre

In [ ]:
movie_tags = tags.groupby('movieId')['tag'].apply(lambda x: " ".join(x.astype(str))).reset_index()
movies = movies.merge(movie_tags, on='movieId', how='left')
movies['tag'] = movies['tag'].fillna('')
movies['genres'] = movies['genres'].str.replace('|', ' ', regex=False)

movies['features'] = movies['genres'] + " " + movies['tag']

0    Adventure Animation Children Comedy Fantasy pi...
1    Adventure Children Fantasy fantasy magic board...
2                             Comedy Romance moldy old
3                                Comedy Drama Romance 
4                              Comedy pregnancy remake
Name: features, dtype: str


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english')

vectors = tfidf.fit_transform(movies['features'])
# Convert text into mathematical vectors.

# ML models only work with numbers.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(vectors)
# Measures how close movies are.

In [12]:
def recommend(movie_name):
    
    idx = movies[movies['title'] == movie_name].index[0]

    distances = similarity[idx]

    movie_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:6]

    for i in movie_list:
        print(movies.iloc[i[0]].title)
recommend("Toy Story (1995)")

Bug's Life, A (1998)
Toy Story 2 (1999)
Guardians of the Galaxy 2 (2017)
Antz (1998)
Adventures of Rocky and Bullwinkle, The (2000)


In [13]:
import pickle

pickle.dump(movies, open('../server/models/movie_list.pkl', 'wb'))
pickle.dump(similarity, open('../server/models/similarity.pkl', 'wb'))